# 03 — Análise Estatística

Objetivo: verificar formalmente, com correlação, teste de hipóteses e regressão,
os padrões observados na EDA sobre escolaridade, remuneração e setor econômico.

Como no restante do projeto, os resultados são interpretados como **associação**,
não causalidade.

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pyarrow.parquet as pq

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

DATA_PROCESSED = Path("../data/processed")
FIGURES = Path("../figures")
arquivo = DATA_PROCESSED / "rais_sp_2025.parquet"

df = pq.ParquetFile(arquivo).read().to_pandas()
df.shape

(14948065, 16)

In [6]:
# Nomes originais -> nomes usados nas análises estatísticas.
# Baseado nas colunas efetivamente gravadas em data/processed/rais_sp_2025.parquet
# (ver 01_preparacao_dados.ipynb, colunas_processamento + colunas derivadas "Escolaridade"/"Setor").
rename_map = {
    "Vl Rem Média Nom": "remuneracao_media",
    "Vl Rem Dezembro Nom": "remuneracao_dezembro",
    "Escolaridade": "escolaridade",
    "Setor": "setor",
    "Idade": "idade",
    "Sexo - Código": "sexo",
    "Qtd Hora Contr": "horas_contratadas",
    "Tempo Emprego": "tempo_emprego",
    "Tipo Vínculo - Código": "tipo_vinculo",
    "Tamanho Estabelecimento - Código": "tamanho_estabelecimento",
    "CBO 2002 Ocupação - Código": "cbo_ocupacao",
}

df = df.rename(columns=rename_map)
df.columns.tolist()

['cbo_ocupacao',
 'CNAE 2.0 Classe - Código',
 'Ind Vínculo Ativo 31/12 - Código',
 'Escolaridade Após 2005 - Código',
 'horas_contratadas',
 'idade',
 'remuneracao_dezembro',
 'remuneracao_media',
 'sexo',
 'tamanho_estabelecimento',
 'tempo_emprego',
 'tipo_vinculo',
 'Ind Trabalho Intermitente - Código',
 'escolaridade',
 'CNAE_Divisao',
 'setor']

In [7]:
# Mesma ordem/rótulos definidos no 01 (mapa_escolaridade) e usados no 02.
# Importante: são os textos EXATOS já gravados na coluna "escolaridade" do parquet.
ordem_escolaridade = [
    "Analfabeto",
    "Até 5º ano incompleto",
    "5º ano completo do Fundamental",
    "6º ao 9º ano do Fundamental",
    "Fundamental completo",
    "Médio incompleto",
    "Médio completo",
    "Superior incompleto",
    "Superior completo",
    "Mestrado",
    "Doutorado",
]

df["escolaridade"] = pd.Categorical(
    df["escolaridade"], categories=ordem_escolaridade, ordered=True
)

# Checagem: nenhum valor deve virar NaN nesse passo (senão, algum rótulo não bateu)
print("Escolaridade não reconhecida:", df["escolaridade"].isna().sum())

# rank ordinal — usado SÓ no Spearman, nunca como variável numérica na regressão
df["escolaridade_rank"] = df["escolaridade"].cat.codes

df["log_remuneracao"] = np.log(df["remuneracao_media"])

df[["escolaridade", "escolaridade_rank", "remuneracao_media", "log_remuneracao", "setor"]].head()

Escolaridade não reconhecida: 0


,escolaridade,escolaridade_rank,remuneracao_media,log_remuneracao,setor
0,Médio completo,6,6560.88,8.788880,Comércio
1,6º ao 9º ano do Fundamental,3,2872.43,7.962914,Serviços
2,Superior completo,8,4239.73,8.352255,Comércio
3,Médio completo,6,3995.73,8.292982,Serviços
4,Superior incompleto,7,2580.32,7.855669,Serviços


## Tratamento de valores ausentes/sentinela nas variáveis de controle

`escolaridade`, `setor` e `remuneracao_media` já foram tratados no 01 (são as
variáveis centrais da população analítica). As variáveis de controle
(`sexo`, `tipo_vinculo`, `tamanho_estabelecimento`, `idade`, `horas_contratadas`,
`tempo_emprego`) ainda não passaram por esse tratamento — seguindo o mesmo
critério documentado no 01 ("tratar códigos classificados como ignorados (-1)
como valores ausentes"), fazemos isso agora, só para as variáveis que entram
no modelo.

In [8]:
colunas_controle_codigo = ["sexo", "tipo_vinculo", "tamanho_estabelecimento"]

for col in colunas_controle_codigo:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))
    print()

# Código -1 = ignorado, mesmo critério já usado no 01 para outras variáveis
for col in colunas_controle_codigo:
    df[col] = df[col].replace(-1, np.nan)

# Checagem rápida de faixas plausíveis (sem excluir nada ainda, só inspecionando)
print(df[["idade", "horas_contratadas", "tempo_emprego"]].describe())

--- sexo ---
sexo
1    8023398
2    6924667
Name: count, dtype: int64

--- tipo_vinculo ---
tipo_vinculo
10    12117061
30     1019024
60      789398
55      175241
50      141159
96      134882
25      132578
31      112709
15       87684
20       77893
35       70853
97       32075
90       31081
75        6207
40        5640
Name: count, dtype: int64

--- tamanho_estabelecimento ---
tamanho_estabelecimento
10    3803161
5     1854981
7     1746419
6     1396033
8     1364530
4     1363627
9     1269979
3     1096698
2     1052637
Name: count, dtype: int64

              idade  horas_contratadas  tempo_emprego
count  1.494806e+07       1.494806e+07   1.494806e+07
mean   3.883151e+01       4.317391e+01   5.729249e+01
std    1.240515e+01       1.079528e+01   8.065483e+01
min    0.000000e+00       0.000000e+00   0.000000e+00
25%    2.900000e+01       4.000000e+01   8.000000e+00
50%    3.800000e+01       4.400000e+01   2.300000e+01
75%    4.800000e+01       4.400000e+01   6.930000e+01
ma

In [9]:
# Contagem de quanto seria perdido se exigíssemos todas as variáveis de controle
colunas_modelo = [
    "escolaridade", "setor", "remuneracao_media", "log_remuneracao",
    "idade", "sexo", "horas_contratadas", "tempo_emprego",
    "tipo_vinculo", "tamanho_estabelecimento",
]

antes = len(df)
depois = len(df.dropna(subset=colunas_modelo))
print(f"Registros antes do dropna nas variáveis de controle: {antes:,}")
print(f"Registros após dropna (usados no modelo):            {depois:,}")
print(f"Percentual removido: {(antes - depois) / antes * 100:.2f}%")

Registros antes do dropna nas variáveis de controle: 14,948,065
Registros após dropna (usados no modelo):            14,948,065
Percentual removido: 0.00%


## Correlação

`escolaridade` é ordinal, então usamos **Spearman** (baseado em ranks), que não
assume distância igual entre níveis — diferente de tratar o código 1-11 como
uma variável numérica contínua (o que distorceria a interpretação).

In [12]:
rho_rem, p_rem = stats.spearmanr(df["escolaridade_rank"], df["remuneracao_media"])
rho_log, p_log = stats.spearmanr(df["escolaridade_rank"], df["log_remuneracao"])

print(f"Spearman (escolaridade x remuneração):      rho={rho_rem:.4f}  p={p_rem:.2e}")
print(f"Spearman (escolaridade x log-remuneração):   rho={rho_log:.4f}  p={p_log:.2e}")

corr_controles = df[["idade", "horas_contratadas", "tempo_emprego", "log_remuneracao"]].corr(
    method="spearman"
)
corr_controles

Spearman (escolaridade x remuneração):      rho=0.4664  p=0.00e+00
Spearman (escolaridade x log-remuneração):   rho=0.4664  p=0.00e+00


,idade,horas_contratadas,tempo_emprego,log_remuneracao
idade,1.000000,-0.066727,0.406735,0.279036
horas_contratadas,-0.066727,1.000000,-0.153078,-0.164055
tempo_emprego,0.406735,-0.153078,1.000000,0.425083
log_remuneracao,0.279036,-0.164055,0.425083,1.000000


## Teste de hipóteses

H0: a remuneração mediana é igual entre todos os grupos. Como a distribuição é
fortemente assimétrica (skew = 7,70, confirmado no 02), usamos o teste não
paramétrico **Kruskal-Wallis**.

In [13]:
grupos_escolaridade = [
    grupo["log_remuneracao"].values
    for _, grupo in df.groupby("escolaridade", observed=True)
]
h_esc, p_esc = stats.kruskal(*grupos_escolaridade)
print(f"Kruskal-Wallis (escolaridade): H={h_esc:.2f}  p={p_esc:.2e}")

grupos_setor = [
    grupo["log_remuneracao"].values
    for _, grupo in df.groupby("setor", observed=True)
]
h_set, p_set = stats.kruskal(*grupos_setor)
print(f"Kruskal-Wallis (setor): H={h_set:.2f}  p={p_set:.2e}")

Kruskal-Wallis (escolaridade): H=4076790.86  p=0.00e+00
Kruskal-Wallis (setor): H=420002.40  p=0.00e+00


## Regressão

Modelo principal: `log(remuneração) ~ escolaridade (categórica) + setor (categórico) + controles`,
com erros-padrão robustos à heterocedasticidade (HC3), esperada em dados salariais.
Amostra aleatória usada para viabilizar o processamento (14,9 milhões de linhas seria
inviável em memória para OLS com muitas dummies).

In [14]:
USE_SAMPLE = True
SAMPLE_SIZE = 300_000

df_modelo = df.dropna(subset=colunas_modelo).copy()

if USE_SAMPLE:
    df_modelo = df_modelo.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

formula_principal = (
    "log_remuneracao ~ C(escolaridade) + C(setor) "
    "+ idade + I(idade**2) + C(sexo) "
    "+ horas_contratadas + tempo_emprego + C(tipo_vinculo) + C(tamanho_estabelecimento)"
)

modelo_principal = smf.ols(formula_principal, data=df_modelo).fit(cov_type="HC3")
print(modelo_principal.summary())

                            OLS Regression Results                            
Dep. Variable:        log_remuneracao   R-squared:                       0.501
Model:                            OLS   Adj. R-squared:                  0.501
Method:                 Least Squares   F-statistic:                     5442.
Date:                Thu, 17 Sep 2026   Prob (F-statistic):               0.00
Time:                        04:01:34   Log-Likelihood:            -2.2297e+05
No. Observations:              300000   AIC:                         4.460e+05
Df Residuals:                  299953   BIC:                         4.465e+05
Df Model:                          46                                         
Covariance Type:                  HC3                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [15]:
coefs_escolaridade = modelo_principal.params.filter(like="C(escolaridade)")
tabela_efeitos = (
    ((np.exp(coefs_escolaridade) - 1) * 100)
    .rename("efeito_percentual_vs_referencia")
    .to_frame()
)
tabela_efeitos.index = [i.split("T.")[-1].strip("]") for i in tabela_efeitos.index]
tabela_efeitos.sort_values("efeito_percentual_vs_referencia")

,efeito_percentual_vs_referencia
Até 5º ano incompleto,3.444293
6º ao 9º ano do Fundamental,5.214437
5º ano completo do Fundamental,6.759971
Fundamental completo,10.707497
Médio incompleto,13.994475
Médio completo,23.476608
Superior incompleto,54.340283
Superior completo,152.812199
Mestrado,290.483795
Doutorado,347.628295


## O retorno da escolaridade varia por setor? (modelo com interação)

Testa formalmente se o padrão do heatmap `remuneracao_escolaridade_setor.png`
(EDA) é uma interação estatisticamente significativa, e não ruído amostral.

In [16]:
formula_interacao = (
    "log_remuneracao ~ C(escolaridade) * C(setor) "
    "+ idade + I(idade**2) + C(sexo) "
    "+ horas_contratadas + tempo_emprego + C(tipo_vinculo) + C(tamanho_estabelecimento)"
)

modelo_interacao = smf.ols(formula_interacao, data=df_modelo).fit(cov_type="HC3")

anova_resultado = anova_lm(modelo_principal, modelo_interacao)
print(anova_resultado)
print(f"\nR² modelo principal:      {modelo_principal.rsquared:.4f}")
print(f"R² modelo com interação:  {modelo_interacao.rsquared:.4f}")

   df_resid           ssr  df_diff     ss_diff          F         Pr(>F)
0  299953.0  77661.096442      0.0         NaN        NaN            NaN
1  299913.0  77467.037280     40.0  194.059162  18.782461  7.939041e-132

R² modelo principal:      0.5007
R² modelo com interação:  0.5020
